# 🏠 Housing Price Prediction — EDA, Preprocessing & Baseline Modelling

**Task type:** Tabular regression  |  **Target:** `median_house_value` (USD)  |  **Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn

This notebook is an end-to-end pipeline:

1. Load & inspect the data
2. **Split first** (so nothing about the test set leaks into any later step)
3. Exploratory data analysis on the *training* set only (univariate, bivariate, correlation, outliers)
4. Cleaning & feature engineering, implemented as scikit-learn `Pipeline`s so every statistic (medians, clipping bounds, scaling parameters, encoder categories) is learned from training data only
5. Train and compare four distinct algorithms with cross-validation
6. Tune the best candidate, evaluate once on the held-out test set (RMSE, MAE, R²)
7. Diagnostics, interpretation, limitations

**Dataset.** The California Housing dataset (1990 US census, one row per census block group, 20,640 rows). It is a good fit for the brief because it contains real-world issues: missing values (`total_bedrooms`), a categorical feature (`ocean_proximity`), heavily skewed count features with outliers, and a right-censored target (values capped at $500,001).

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"primary": "#2a6fdb", "accent": "#e8590c", "neutral": "#868e96"}
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold", "axes.titlesize": 12})

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 2. Load the data

The loader looks for a local copy first (`data/housing.csv` or `housing.csv`, which is what the GitHub repo contains) and otherwise downloads the same file from a public mirror, so the notebook runs unchanged on Colab, Kaggle or locally.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
LOCAL_CANDIDATES = [Path("data/housing.csv"), Path("housing.csv")]

local = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
source = str(local) if local else DATA_URL
df = pd.read_csv(source)
print(f"Loaded from: {source}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

### 2.1 Column dictionary

| Column | Meaning |
|---|---|
| `longitude`, `latitude` | Block-group location |
| `housing_median_age` | Median age of houses in the block group (years) |
| `total_rooms`, `total_bedrooms` | Totals across the block group |
| `population`, `households` | People / households in the block group |
| `median_income` | Median household income (tens of thousands of USD) |
| `ocean_proximity` | Categorical: distance band to the ocean/bay |
| **`median_house_value`** | **Target**: median house value (USD) |

## 3. First look: types, missing values, duplicates

This step only *inspects*. We do not fill or drop anything yet, because any data-dependent fix must be learned from the training split alone.

In [ ]:
df.info()
print("\nDuplicate rows:", df.duplicated().sum())

missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).query("missing > 0")
print("\nColumns with missing values:")
display(missing)

print("\nCategorical levels:")
display(df["ocean_proximity"].value_counts())

In [ ]:
df.describe().T

**What to notice**

- Only `total_bedrooms` has missing values (about 1% of rows). Because it is so small and numeric, **median imputation** (robust to the skew we are about to see) is appropriate, and dropping rows would waste data for no benefit.
- `max` values of `total_rooms`, `total_bedrooms`, `population` and `households` are many multiples of their medians, i.e. heavy right skew with extreme outliers.
- `ocean_proximity` is nominal (no natural order), so it needs **one-hot encoding**, not integer codes (integer codes would impose a false ordering on the linear model).
- `ISLAND` has only a handful of rows; we will let the encoder handle unseen/rare categories safely.

## 4. Train / test split (done *before* EDA and preprocessing)

To avoid **data leakage**, the test set is carved off now and never touched until the final evaluation. Median income is by far the strongest predictor, so I split **stratified on income bands**. That keeps the income distribution (and hence the difficulty of the problem) the same in train and test, which makes the final score more trustworthy than a purely random split. The helper column used for stratification is dropped immediately afterwards.

In [ ]:
TARGET = "median_house_value"

income_band = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5],
)

train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=income_band
)

X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test,  y_test  = test_df.drop(columns=TARGET),  test_df[TARGET]

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print("\nIncome-band share (train vs test) - should match closely:")
shares = pd.concat(
    [income_band.loc[train_df.index].value_counts(normalize=True).sort_index().rename("train"),
     income_band.loc[test_df.index].value_counts(normalize=True).sort_index().rename("test")],
    axis=1,
)
display(shares)

## 5. Exploratory Data Analysis (training data only)

All plots below use `train_df`. The test set stays untouched so that nothing we "learn" by eye can leak into modelling decisions in a way that inflates the final score.

### 5.1 Univariate analysis

**Target first.** Its shape tells us how errors should be interpreted and whether a transform might be needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(train_df[TARGET], bins=60, color=PALETTE["primary"], ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel("median_house_value (USD)")

sns.boxplot(x=train_df[TARGET], color=PALETTE["primary"], ax=axes[1])
axes[1].set_title("Target box plot")
axes[1].set_xlabel("median_house_value (USD)")
plt.tight_layout(); plt.show()

cap = train_df[TARGET].max()
n_cap = (train_df[TARGET] == cap).sum()
print(f"Skewness of target: {train_df[TARGET].skew():.2f}")
print(f"Maximum value {cap:,.0f} appears {n_cap} times ({n_cap/len(train_df):.1%} of training rows) -> censored/capped target")

**Finding.** The target is right-skewed and has a visible spike at the maximum value: the census data was **capped at about $500,000**, so the most expensive blocks are all recorded as the same number. Two consequences:

1. Models cannot learn the true price above the cap, so errors on very expensive blocks will be larger and systematic (we will see this in the residual plot).
2. I deliberately keep these rows rather than deleting them. Removing them from training would make the model look better on a cleaned test distribution but worse on real data that includes them; the limitation is documented instead.

In [ ]:
num_cols_raw = X_train.select_dtypes(include="number").columns.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), num_cols_raw):
    sns.histplot(X_train[col], bins=50, color=PALETTE["primary"], ax=ax)
    ax.set_title(f"{col}  (skew {X_train[col].skew():.1f})")
    ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols_raw):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
order = X_train["ocean_proximity"].value_counts().index
sns.countplot(data=X_train, y="ocean_proximity", order=order, color=PALETTE["primary"], ax=ax)
ax.set_title("ocean_proximity - category counts")
plt.tight_layout(); plt.show()

**Finding.** `total_rooms`, `total_bedrooms`, `population` and `households` are strongly right-skewed with long tails, so raw magnitudes mostly reflect *how big the block group is*, not how expensive it is. `housing_median_age` also shows a spike at its maximum (another cap). `median_income` is moderately right-skewed. The category `ISLAND` is extremely rare. These observations drive the preprocessing choices in Section 6: ratio features to remove the size effect, percentile clipping for the tails, and scaling for the linear model.

### 5.2 Outlier inspection

I use the 1.5 x IQR rule only as a *diagnostic* to quantify how many points are extreme in each feature.

In [ ]:
def iqr_outlier_share(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean()

outlier_table = pd.Series({c: iqr_outlier_share(X_train[c]) for c in num_cols_raw}, name="share_flagged_by_IQR")
display(outlier_table.sort_values(ascending=False).to_frame().style.format("{:.1%}"))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, col in zip(axes, ["total_rooms", "total_bedrooms", "population", "households"]):
    sns.boxplot(y=X_train[col], color=PALETTE["accent"], ax=ax)
    ax.set_title(col); ax.set_ylabel("")
plt.tight_layout(); plt.show()

**Finding.** The four "count" features have a large number of flagged points and a few extreme values. These are *real* (very large census blocks), not data-entry errors, so deleting them would discard legitimate rows. Instead I will **winsorise (clip) at the 1st and 99th percentiles learned on the training set**, which limits the leverage of extreme points on the linear model while keeping every row.

### 5.3 Bivariate analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["median_income", "housing_median_age", "total_rooms"]):
    sns.scatterplot(data=train_df.sample(4000, random_state=RANDOM_STATE), x=col, y=TARGET,
                    alpha=0.35, s=14, color=PALETTE["primary"], ax=ax)
    ax.set_title(f"{TARGET} vs {col}")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
order = train_df.groupby("ocean_proximity")[TARGET].median().sort_values().index
sns.boxplot(data=train_df, x="ocean_proximity", y=TARGET, order=order, color=PALETTE["primary"], ax=ax)
ax.set_title("House value by ocean_proximity")
plt.tight_layout(); plt.show()

display(train_df.groupby("ocean_proximity")[TARGET].agg(["count", "median", "mean"]).sort_values("median"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
sc = ax.scatter(train_df["longitude"], train_df["latitude"], c=train_df[TARGET],
                s=train_df["population"] / 100, alpha=0.4, cmap="viridis")
plt.colorbar(sc, label="median_house_value (USD)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Geography: colour = price, size = population")
plt.tight_layout(); plt.show()

**Findings.**

- `median_income` shows the clearest positive, roughly linear relationship with price (and the horizontal band of points at the cap is visible).
- `housing_median_age` and `total_rooms` show little pattern on their own.
- `ocean_proximity` matters: coastal and bay categories have clearly higher medians than `INLAND`, so location is informative.
- The map confirms that price is strongly **geographic** (high near the coast and the big metro areas). Latitude/longitude therefore carry non-linear signal that a plain linear model cannot exploit but tree-based models can. This is a hypothesis we will test in the model comparison.

### 5.4 Correlation analysis

In [ ]:
corr = train_df.select_dtypes(include="number").corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.5, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix (training data)")
plt.tight_layout(); plt.show()

print("Correlation with target:")
display(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_frame("pearson_r"))

**Findings.**

- `median_income` is by far the strongest linear correlate of price.
- `total_rooms`, `total_bedrooms`, `population` and `households` are **highly inter-correlated** (they all scale with block-group size). This **multicollinearity** destabilises linear-model coefficients and tells us the raw totals are redundant.
- Their individual correlation with price is weak, which suggests *ratios* (rooms per household, etc.) will be more informative than the raw totals.

## 6. Preprocessing & feature engineering

Everything is wrapped in scikit-learn `Pipeline`/`ColumnTransformer` objects. The pipeline is **fit on training data only**, then merely *applied* to validation folds and the test set, so imputation medians, clipping bounds, scaler statistics and encoder categories can never see held-out data. Cross-validation re-fits the whole pipeline inside every fold for the same reason.

| Step | Choice | Why |
|---|---|---|
| Feature engineering | `rooms_per_household`, `bedrooms_per_room`, `population_per_household` | Removes the block-size effect behind the multicollinearity; ratios describe housing *character* |
| Missing values | Median imputation | Robust to skew; only ~1% missing |
| Outliers | Clip to training 1st-99th percentile | Keeps every row, limits extreme leverage |
| Scaling | `StandardScaler` for linear model only | Linear models are scale-sensitive; tree models are invariant to monotone scaling, so it is skipped for them |
| Encoding | One-hot (`handle_unknown="ignore"`) | `ocean_proximity` is nominal |

In [ ]:
class HousingFeatureAdder(BaseEstimator, TransformerMixin):
    # Adds ratio features. Stateless, so there is nothing to leak.
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["rooms_per_household"] = X["total_rooms"] / X["households"]
        X["bedrooms_per_room"] = X["total_bedrooms"] / X["total_rooms"]   # NaN stays NaN -> imputed later
        X["population_per_household"] = X["population"] / X["households"]
        return X


class QuantileClipper(BaseEstimator, TransformerMixin):
    # Winsorise each column to [lower_q, upper_q] percentiles learned in fit() (i.e. from training data only).
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(X, self.upper_q, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


NUMERIC = [
    "longitude", "latitude", "housing_median_age", "total_rooms", "total_bedrooms",
    "population", "households", "median_income",
    "rooms_per_household", "bedrooms_per_room", "population_per_household",   # engineered
]
CATEGORICAL = ["ocean_proximity"]


def build_preprocessor(scale: bool, clip: bool) -> ColumnTransformer:
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if clip:
        num_steps.append(("clip", QuantileClipper(0.01, 0.99)))
    if scale:
        num_steps.append(("scale", StandardScaler()))

    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUMERIC),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORICAL),
    ])


def make_model_pipeline(estimator, scale: bool, clip: bool) -> Pipeline:
    return Pipeline([
        ("features", HousingFeatureAdder()),
        ("prep", build_preprocessor(scale=scale, clip=clip)),
        ("model", estimator),
    ])

In [ ]:
# Quick sanity check on a tiny sample: shape, no NaNs left, one-hot width
probe = make_model_pipeline(LinearRegression(), scale=True, clip=True)
probe_X = probe[:-1].fit_transform(X_train)
print("Transformed training matrix:", probe_X.shape)
print("NaNs remaining:", int(np.isnan(probe_X).sum()))
print("Column means (should be ~0 for scaled numerics):", np.round(probe_X[:, :3].mean(axis=0), 3))

The check confirms the pipeline produces a complete (no-NaN) numeric matrix. Linear models get **clip + scale**; tree models get **clip only** (clipping is harmless to them, scaling is unnecessary), keeping each model on the preprocessing it actually benefits from.

## 7. Model training & comparison

Four distinct algorithms, chosen to span the bias/variance spectrum:

| Model | Role / justification |
|---|---|
| **Linear Regression** | Interpretable **baseline**; shows how much a purely linear, additive model can explain |
| **Decision Tree** | Captures non-linearity and interactions (e.g. latitude x longitude); high variance, so it shows why ensembling helps |
| **Random Forest** | Bagged trees: lowers variance, strong default for tabular data, little tuning needed |
| **Hist. Gradient Boosting** | Boosted trees: lowers bias, usually the strongest on tabular data, fast on 16k+ rows |

**Protocol.** 5-fold cross-validation on the *training set only*; models are compared on cross-validated RMSE, MAE and R². The test set is not used for model selection.

In [ ]:
models = {
    "Linear Regression": make_model_pipeline(LinearRegression(), scale=True, clip=True),
    "Decision Tree": make_model_pipeline(
        DecisionTreeRegressor(max_depth=10, min_samples_leaf=10, random_state=RANDOM_STATE),
        scale=False, clip=True),
    "Random Forest": make_model_pipeline(
        RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
        scale=False, clip=True),
    "Hist Gradient Boosting": make_model_pipeline(
        HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        scale=False, clip=True),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1, return_train_score=True)
    rows.append({
        "model": name,
        "cv_RMSE": -res["test_rmse"].mean(),
        "cv_RMSE_std": res["test_rmse"].std(),
        "cv_MAE": -res["test_mae"].mean(),
        "cv_R2": res["test_r2"].mean(),
        "train_R2": res["train_r2"].mean(),
    })
    print(f"done: {name}")

cv_results = pd.DataFrame(rows).set_index("model").sort_values("cv_RMSE")
cv_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = cv_results.index
sns.barplot(x=cv_results["cv_RMSE"], y=order, xerr=cv_results["cv_RMSE_std"], color=PALETTE["primary"], ax=axes[0])
axes[0].set_title("Cross-validated RMSE (lower is better)"); axes[0].set_xlabel("USD")
sns.barplot(x=cv_results["cv_R2"], y=order, color=PALETTE["accent"], ax=axes[1])
axes[1].set_title("Cross-validated R² (higher is better)"); axes[1].set_xlabel("R²"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

**How to read this.**

- The gap between `train_R2` and `cv_R2` is the overfitting signal: a large gap (typical for an unconstrained Random Forest) means the model memorises training data; a small gap (Linear Regression) means it is limited by bias rather than variance.
- The **linear baseline** underfits because price depends non-linearly on location and income; the tree-based models can capture those interactions and are expected to clearly beat it.
- Boosting and bagging typically beat a single tree, which has high variance.

The numbers in the table above are the evidence. The best model by CV RMSE is carried forward to tuning.

## 8. Hyper-parameter tuning

I tune the best-scoring ensemble by randomised search (cross-validated on the training set), which is cheaper than a full grid and usually finds comparable settings. If a plain Linear Regression ever came out best, there would be nothing to tune.

In [ ]:
param_spaces = {
    "Random Forest": {
        "model__n_estimators": [200, 300, 400],
        "model__max_depth": [None, 15, 25],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.33, 0.5, 0.8, 1.0],
    },
    "Hist Gradient Boosting": {
        "model__learning_rate": [0.03, 0.05, 0.1, 0.2],
        "model__max_iter": [200, 400, 600],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__min_samples_leaf": [10, 20, 40],
        "model__l2_regularization": [0.0, 0.1, 1.0],
    },
}

best_name = cv_results.index[0]
print("Best by CV RMSE:", best_name)

# Always tune the best *tunable* ensemble
tune_name = next(n for n in cv_results.index if n in param_spaces)   # first (best-ranked) model that has a search space
print("Tuning:", tune_name)

search = RandomizedSearchCV(
    models[tune_name], param_distributions=param_spaces[tune_name],
    n_iter=12, cv=3, scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
search.fit(X_train, y_train)

print("\nBest params:", {k.replace("model__", ""): v for k, v in search.best_params_.items()})
print(f"Tuned CV RMSE (3-fold): {-search.best_score_:,.0f}")
tuned_model = search.best_estimator_

## 9. Final evaluation on the held-out test set

This is the **first and only** time the test set is used. Each model is refit on the full training set and scored once on the test set. Model *selection* was done with cross-validation above, so the test scores are an unbiased estimate of generalisation.

In [ ]:
def regression_report(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

final_models = dict(models)
final_models[f"{tune_name} (tuned)"] = tuned_model

test_rows, preds = {}, {}
for name, pipe in final_models.items():
    if name != f"{tune_name} (tuned)":
        pipe = pipe.fit(X_train, y_train)          # tuned model is already refit by the search
    preds[name] = pipe.predict(X_test)
    test_rows[name] = regression_report(y_test, preds[name])

test_results = pd.DataFrame(test_rows).T.sort_values("RMSE")
display(test_results)

base = test_results.loc["Linear Regression", "RMSE"]
best_final = test_results.index[0]
print(f"\nBest on test set: {best_final}")
print(f"RMSE {test_results.loc[best_final, 'RMSE']:,.0f} USD  |  MAE {test_results.loc[best_final, 'MAE']:,.0f} USD  |  R² {test_results.loc[best_final, 'R2']:.3f}")
print(f"RMSE reduction vs linear baseline: {1 - test_results.loc[best_final, 'RMSE'] / base:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = test_results.reset_index().melt(id_vars="index", value_vars=["RMSE", "MAE"])
sns.barplot(data=plot_df, y="index", x="value", hue="variable",
            palette=[PALETTE["primary"], PALETTE["accent"]], ax=ax)
ax.set_title("Test-set error by model (lower is better)")
ax.set_xlabel("USD"); ax.set_ylabel(""); ax.legend(title="")
plt.tight_layout(); plt.show()

## 10. Diagnostics of the best model

In [ ]:
best_pred = preds[best_final]
resid = y_test - best_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

axes[0].scatter(y_test, best_pred, s=8, alpha=0.3, color=PALETTE["primary"])
lims = [0, max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, color=PALETTE["accent"], lw=1.5, label="perfect prediction")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].set_title("Predicted vs actual"); axes[0].legend()

axes[1].scatter(best_pred, resid, s=8, alpha=0.3, color=PALETTE["primary"])
axes[1].axhline(0, color=PALETTE["accent"], lw=1.5)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual (actual - predicted)"); axes[1].set_title("Residuals vs predicted")

sns.histplot(resid, bins=60, color=PALETTE["primary"], ax=axes[2])
axes[2].set_title("Residual distribution"); axes[2].set_xlabel("Residual (USD)")
plt.suptitle(f"{best_final}", y=1.02)
plt.tight_layout(); plt.show()

capped = (y_test == y_test.max())
print(f"Mean residual on capped blocks: {resid[capped].mean():,.0f} USD  vs  non-capped: {resid[~capped].mean():,.0f} USD")

**Reading the plots.**

- Points hug the diagonal in the predicted-vs-actual plot where data is dense. A **horizontal band of actual values at the cap** (prediction below the truth) is the signature of the censored target discussed in Section 5.1: the model cannot know that those blocks are worth *more* than the recorded $500k, so the residuals there are positive (under-prediction).
- Residual spread increases with the predicted price (heteroscedasticity): errors are proportionally larger for expensive houses. A log-transformed target is a natural improvement (see Next steps).
- The residual histogram should be roughly centred on zero with heavier tails than a normal distribution.

In [ ]:
# Permutation importance on the test set, computed on the ORIGINAL input columns (works through the whole pipeline)
perm = permutation_importance(
    final_models[best_final], X_test, y_test,
    n_repeats=5, random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
imp = (pd.DataFrame({"feature": X_test.columns, "rmse_increase": perm.importances_mean, "std": perm.importances_std})
         .sort_values("rmse_increase", ascending=True))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(imp["feature"], imp["rmse_increase"], xerr=imp["std"], color=PALETTE["primary"])
ax.set_title("Permutation importance (increase in test RMSE when feature is shuffled)")
ax.set_xlabel("RMSE increase (USD)")
plt.tight_layout(); plt.show()

**Interpretation.** Shuffling `median_income` damages the model most, followed by the location features (`latitude`, `longitude`, `ocean_proximity`). This agrees with the EDA: income and geography are the main drivers of price, and the model learned that from the data rather than from anything we hard-coded. Features near zero importance (for example raw room/bedroom totals, which are redundant with the engineered ratios) contribute little individually.

## 11. Summary of findings

In [ ]:
summary = test_results.round(3)
print("Final test-set comparison (sorted by RMSE):")
display(summary)
b = test_results.loc[best_final]
print(f"\n=> {best_final}: R² = {b['R2']:.3f}, typical error (MAE) ≈ ${b['MAE']:,.0f}, RMSE ≈ ${b['RMSE']:,.0f}")
print(f"=> RMSE is {1 - b['RMSE'] / base:.1%} lower than the linear baseline.")

### Takeaways

1. **Data issues handled:** one numeric column with missing values (median-imputed), one nominal categorical (one-hot encoded), heavy right-skew and outliers in count features (percentile clipping), and multicollinearity among size-related features (replaced in effect by ratio features).
2. **EDA drove the modelling:** the strongest signals are median income and geography; relationships with location are non-linear, which motivated tree-based models alongside a linear baseline.
3. **Model comparison:** non-linear ensemble models should clearly outperform the linear baseline and a single decision tree; the exact numbers are in the tables above, selected on cross-validation and confirmed once on the untouched test set.
4. **No leakage:** the split happens first, all fitted statistics live inside pipelines fit on training data only, cross-validation refits the pipeline per fold, and the test set is used a single time.

### Limitations

- **Censored target** (cap near $500k) caps achievable accuracy for the most expensive blocks.
- **Data is from 1990**, so absolute prices are not meaningful today; the *workflow* is what is being demonstrated.
- Block-group *medians*, not individual houses; predictions describe areas, not properties.
- Random (not spatial) cross-validation: neighbouring blocks in different folds can make scores slightly optimistic.

### Next steps

- Model `log(median_house_value)` to stabilise the variance of residuals.
- Add spatial features (distance to major cities / coast, spatial clustering) and use spatially-blocked cross-validation.
- Try XGBoost / LightGBM and a wider search, and add prediction intervals (e.g. quantile regression).